# Databricks/PySpark dla Data Analyst/Engineer — Moduł 10: Delta Lake -- podstawy

Ten moduł jest inny niż poprzednie: przez Moduły 1-9 każdy kod dało się uruchomić
lokalnie. **Delta Lake** wymaga pobrania dodatkowych bibliotek Javy z internetu przy
pierwszym użyciu (albo natywnego wsparcia, jak w Databricks) -- w niektórych środowiskach
lokalnych (w tym w środowisku, w którym przygotowano ten kurs) pobranie tych bibliotek
jest zablokowane przez politykę sieciową. Kod w tym module jest poprawny i gotowy do
uruchomienia w **Databricks Free Edition** (gdzie Delta Lake jest wbudowane natywnie,
bez żadnej konfiguracji) -- oznaczamy to wyraźnie przy każdej sekcji.

> ⚠️ **Ten moduł nie był uruchamiany na tym samym silniku wykonawczym co Moduły 1-9**
>
> W przeciwieństwie do wcześniejszych modułów, kod w tym notatniku NIE został automatycznie zweryfikowany przez wykonanie w tym samym środowisku, w którym przygotowano kurs (dostęp do repozytorium Maven, skąd Spark pobiera bibliotekę Delta Lake, jest tam zablokowany). Składnia jest dokładnie zgodna z oficjalną dokumentacją Delta Lake -- ale jeśli uruchamiasz ten notatnik na WŁASNYM komputerze z pełnym dostępem do internetu, pierwsze uruchomienie może potrwać dłużej (Spark pobierze potrzebne pliki `.jar`). W Databricks Free Edition wszystko działa od razu, bez żadnego pobierania.

## Spis treści
1. [Dlaczego Delta Lake -- problemy ze zwykłym Parquetem](#sec1)
2. [Transakcje ACID i _delta_log](#sec2)
3. [Time travel: podróżowanie w historii danych](#sec3)
4. [MERGE INTO -- upsert w jednym poleceniu](#sec4)
5. [Ewolucja schematu](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Dlaczego Delta Lake -- problemy ze zwykłym Parquetem

W Module 6 zapisywaliśmy dane do zwykłego Parquetu. To dobry format, ale ma poważne
ograniczenia przy pracy zespołowej i w produkcyjnych potokach danych:

- **Brak transakcji ACID** -- jeśli zapis do Parquetu zostanie przerwany w połowie
  (awaria, restart klastra), folder może zostać w NIESPÓJNYM stanie -- część plików
  zapisana, część nie.
- **Brak egzekwowania schematu** -- nic nie stoi na przeszkodzie, żeby ktoś zapisał do
  tego samego folderu dane z INNYM schematem (np. inny typ kolumny), psując wszystkie
  przyszłe odczyty.
- **Brak historii** -- gdy nadpiszesz dane (`mode("overwrite")`), STARA wersja znika
  bezpowrotnie. Nie da się "cofnąć" do stanu sprzed błędnego zapisu.
- **Brak prawdziwego UPDATE/DELETE** -- Parquet to format WYŁĄCZNIE do zapisu/odczytu
  całych plików -- nie ma pojęcia "zmień ten jeden wiersz", trzeba przepisać cały plik.

**Delta Lake** to warstwa NAD Parquetem (pliki danych to nadal zwykłe pliki `.parquet`),
która dokłada transakcyjny log rozwiązujący wszystkie powyższe problemy.

> ⚡ **Delta Lake to nie inny format pliku -- to Parquet + log transakcji**
>
> Kluczowe do zrozumienia: dane w tabeli Delta są fizycznie zapisane jako zwykłe pliki Parquet. To, co dokłada Delta Lake, to folder `_delta_log/` obok nich, zawierający historię wszystkich operacji. Dzięki temu Delta Lake jest wstecznie kompatybilny z narzędziami czytającymi Parquet, a jednocześnie dodaje funkcjonalność bazodanową.

<a id="sec2"></a>
## 2. Transakcje ACID i _delta_log

Zapis do formatu `delta` (zamiast `parquet`) tworzy w docelowym folderze podfolder
`_delta_log/` -- sekwencję ponumerowanych plików JSON (`00000000000000000000.json`,
`00000000000000000001.json`, ...), z których każdy opisuje JEDNĄ transakcję: które
pliki Parquet zostały dodane, które usunięte, jaki był schemat w tym momencie.

In [ ]:
# Zapis w formacie delta -- skladnia niemal identyczna z Modulu 6, tylko inny format
klienci.write.format("delta").mode("overwrite").save("dane_delta/klienci_delta")

# Odczyt -- rowniez niemal identyczny jak dla parquet
klienci_delta = spark.read.format("delta").load("dane_delta/klienci_delta")
klienci_delta.show(5)


In [ ]:
# DeltaTable -- specjalny obiekt do operacji specyficznych dla Delta (historia, MERGE, ...)
from delta.tables import DeltaTable

tabela_klienci = DeltaTable.forPath(spark, "dane_delta/klienci_delta")
tabela_klienci.toDF().show(5)


> ⚠️ **Każdy zapis to NOWA transakcja, nie nadpisanie w miejscu**
>
> Nawet `mode("overwrite")` w Delta Lake NIE kasuje fizycznie starych plików od razu -- dopisuje nowy wpis do `_delta_log/`, mówiący 'stare pliki są teraz nieaktywne, użyj tych nowych'. To właśnie umożliwia time travel z sekcji 3 -- stare pliki Parquet fizycznie wciąż leżą na dysku, dopóki nie uruchomisz komendy `VACUUM` (czyszczącej nieużywane, stare pliki po przekroczeniu okresu retencji).

<a id="sec3"></a>
## 3. Time travel: podróżowanie w historii danych

Ponieważ każda zmiana w tabeli Delta to osobny, ponumerowany wpis w logu, można odczytać
DOWOLNĄ wcześniejszą wersję tabeli -- po numerze wersji albo po znaczniku czasu.

In [ ]:
# Historia wszystkich zmian tabeli
tabela_klienci.history().select("version", "timestamp", "operation").show(truncate=False)


In [ ]:
# Symulujemy druga zmiane w tabeli, zeby miec co przegladac w historii
polowa_klientow = klienci.filter(klienci.klient_id <= 150)
polowa_klientow.write.format("delta").mode("overwrite").save("dane_delta/klienci_delta")

# Wersja 0 (sprzed nadpisania) -- odczyt przez opcje versionAsOf
wersja_0 = spark.read.format("delta").option("versionAsOf", 0).load("dane_delta/klienci_delta")
print(f"Wersja 0: {wersja_0.count()} wierszy")

# Najnowsza wersja
najnowsza = spark.read.format("delta").load("dane_delta/klienci_delta")
print(f"Najnowsza wersja: {najnowsza.count()} wierszy")


In [ ]:
# Ten sam time travel w Spark SQL (SELECT ... VERSION AS OF / TIMESTAMP AS OF)
spark.sql("""
    SELECT COUNT(*) AS liczba_wierszy
    FROM delta.`dane_delta/klienci_delta` VERSION AS OF 0
""").show()


> 🧱 **RESTORE -- przywracanie tabeli do starszej wersji NA STAŁE**
>
> Odczyt `VERSION AS OF`/`versionAsOf` tylko PODGLĄDA starą wersję -- nie zmienia bieżącego stanu tabeli. Jeśli chcesz naprawdę PRZYWRÓCIĆ tabelę do wcześniejszej wersji (np. po przypadkowym błędnym zapisie), służy do tego `RESTORE TABLE ... TO VERSION AS OF ...` -- to też tworzy nowy wpis w historii, więc samo `RESTORE` też można później cofnąć.

<a id="sec4"></a>
## 4. MERGE INTO -- upsert w jednym poleceniu

`MERGE INTO` to najważniejsza operacja Delta Lake dla potoków danych: w JEDNYM poleceniu
łączy tabelę docelową ze źródłem nowych/zmienionych danych i decyduje, dla każdego
wiersza, czy go **zaktualizować** (jeśli już istnieje), **wstawić** (jeśli jest nowy),
albo (opcjonalnie) **usunąć**. To dokładnie operacja znana jako *upsert*.

In [ ]:
# Nowe dane: aktualizacja miasta dla czesci klientow + zupelnie nowi klienci
from pyspark.sql import Row

nowe_dane_klientow = spark.createDataFrame([
    Row(klient_id=1, miasto="Wroclaw", segment="Premium", dni_od_rejestracji=500),
    Row(klient_id=301, miasto="Poznan", segment="Nowy", dni_od_rejestracji=1),
    Row(klient_id=302, miasto="Gdansk", segment="Nowy", dni_od_rejestracji=1),
])

tabela_klienci.alias("cel").merge(
    nowe_dane_klientow.alias("zrodlo"),
    "cel.klient_id = zrodlo.klient_id"
).whenMatchedUpdate(set={
    "miasto": "zrodlo.miasto",
    "segment": "zrodlo.segment",
}).whenNotMatchedInsertAll().execute()

tabela_klienci.toDF().orderBy("klient_id").show(5)


> ⚡ **MERGE INTO to samo w Spark SQL**
>
> Dokładnie ta sama operacja w czystym SQL-u wygląda tak:

```sql
MERGE INTO klienci_delta AS cel
USING nowe_dane_klientow AS zrodlo
ON cel.klient_id = zrodlo.klient_id
WHEN MATCHED THEN UPDATE SET miasto = zrodlo.miasto, segment = zrodlo.segment
WHEN NOT MATCHED THEN INSERT *
```

Bez Delta Lake, ten sam efekt wymagałby ręcznego napisania: odczytaj całą starą tabelę, znajdź niepasujące wiersze, połącz z nowymi, zapisz WSZYSTKO od nowa -- znacznie więcej kodu i dużo większy koszt (przepisanie całej tabeli zamiast tylko zmienionych plików).

<a id="sec5"></a>
## 5. Ewolucja schematu

Co się dzieje, gdy nowe dane mają DODATKOWĄ kolumnę, której nie było w oryginalnej
tabeli? Domyślnie Delta Lake (podobnie jak Parquet) rzuci błąd niezgodności schematu --
ale można to świadomie włączyć przez opcję `mergeSchema`.

In [ ]:
from pyspark.sql import functions as F

klienci_z_nowa_kolumna = klienci.withColumn("newsletter", F.lit(True))

# Bez mergeSchema -- to rzuciloby AnalysisException (niezgodnosc schematu)
klienci_z_nowa_kolumna.write.format("delta").mode("append").option(
    "mergeSchema", "true"
).save("dane_delta/klienci_delta")

spark.read.format("delta").load("dane_delta/klienci_delta").printSchema()


> ⚠️ **mergeSchema to świadoma decyzja, nie ustawienie 'na zawsze włączone'**
>
> Włączanie `mergeSchema` domyślnie dla WSZYSTKICH zapisów bywa ryzykowne -- literówka w nazwie kolumny w kodzie źródłowym (np. `kliet_id` zamiast `klient_id`) zamiast rzucić od razu błąd, po cichu DODA nową, niechcianą kolumnę do tabeli. Włączaj `mergeSchema` świadomie, tylko tam, gdzie faktycznie spodziewasz się ewolucji schematu (np. nowe pole dodane przez zespół produkujący dane źródłowe).

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- dlaczego Delta Lake powstał -- ograniczenia zwykłego Parquetu (brak ACID, brak egzekwowania schematu, brak historii, brak update/delete),
- `_delta_log/` -- transakcyjny log JSON leżący pod każdą tabelą Delta,
- time travel: `VERSION AS OF`/`versionAsOf`, `TIMESTAMP AS OF`, `.history()`, `RESTORE`,
- `MERGE INTO` -- upsert (insert + update + opcjonalnie delete) w jednym poleceniu,
- ewolucję schematu przez `mergeSchema`.

Pamiętaj: ten moduł jest w pełni gotowy do uruchomienia w **Databricks Free Edition** --
tam cała powyższa składnia zadziała bez żadnej dodatkowej konfiguracji.

### 📝 Ćwiczenie 1: Zapis i odczyt w formacie Delta

Zapisz `produkty` w formacie `delta` do folderu `dane_delta/produkty_delta`, a następnie odczytaj z powrotem i wypisz `count()`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
produkty.write.format("delta").mode("overwrite").save("dane_delta/produkty_delta")
odczytane = spark.read.format("delta").load("dane_delta/produkty_delta")
print(f"Liczba wierszy: {odczytane.count()}")
```

</details>

### 📝 Ćwiczenie 2: Przegląd historii

Dla tabeli `produkty_delta` z ćwiczenia 1, wykonaj jeszcze jeden zapis (`mode("append")` z dowolnym podzbiorem `produkty`), a następnie wypisz `.history()` -- ile wersji ma teraz tabela?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from delta.tables import DeltaTable

produkty.filter(produkty.produkt_id <= 10).write.format("delta").mode("append").save(
    "dane_delta/produkty_delta"
)

tabela = DeltaTable.forPath(spark, "dane_delta/produkty_delta")
tabela.history().select("version", "timestamp", "operation").show()
```

</details>

### 📝 Ćwiczenie 3: Odczyt starszej wersji

Odczytaj WERSJĘ 0 tabeli `produkty_delta` (sprzed zapisu z ćwiczenia 2) przez `option("versionAsOf", 0)` i porównaj `count()` z najnowszą wersją.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
wersja_0 = spark.read.format("delta").option("versionAsOf", 0).load(
    "dane_delta/produkty_delta"
)
najnowsza = spark.read.format("delta").load("dane_delta/produkty_delta")
print(f"Wersja 0: {wersja_0.count()}, najnowsza: {najnowsza.count()}")
```

</details>

### 📝 Ćwiczenie 4: Prosty MERGE INTO

Dla tabeli `produkty_delta`, wykonaj `MERGE INTO`, który podnosi `cena` o 10% dla produktu o `produkt_id = 1` (jeśli istnieje), a jeśli `produkt_id = 999` nie istnieje -- wstawia go jako nowy produkt.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql import Row

nowe_dane = spark.createDataFrame([
    Row(produkt_id=1, kategoria="Elektronika", cena=None),
    Row(produkt_id=999, kategoria="Elektronika", cena=99.99),
])

tabela.alias("cel").merge(
    nowe_dane.alias("zrodlo"), "cel.produkt_id = zrodlo.produkt_id"
).whenMatchedUpdate(set={"cena": "cel.cena * 1.1"}).whenNotMatchedInsertAll().execute()

tabela.toDF().filter("produkt_id IN (1, 999)").show()
```

</details>

> 🔥 **Wyzwanie: pełny cykl życia tabeli Delta**
>
> Zbuduj pełny scenariusz: (1) zapisz `zamowienia` jako tabelę Delta; (2) wykonaj `MERGE INTO`, który zmienia status WSZYSTKICH zamówień klienta o `klient_id = 1` na `'Anulowane'`; (3) sprawdź `.history()` -- powinny być DWIE wersje (zapis + MERGE); (4) odczytaj WERSJĘ 0 i porównaj status zamówień klienta 1 przed i po MERGE-u.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from delta.tables import DeltaTable
from pyspark.sql import functions as F

zamowienia.write.format("delta").mode("overwrite").save("dane_delta/zamowienia_delta")
tabela_zamowien = DeltaTable.forPath(spark, "dane_delta/zamowienia_delta")

klient_1_nowe = zamowienia.filter(zamowienia.klient_id == 1).withColumn(
    "status", F.lit("Anulowane")
)

tabela_zamowien.alias("cel").merge(
    klient_1_nowe.alias("zrodlo"), "cel.zamowienie_id = zrodlo.zamowienie_id"
).whenMatchedUpdate(set={"status": "zrodlo.status"}).execute()

tabela_zamowien.history().select("version", "operation").show()

przed = spark.read.format("delta").option("versionAsOf", 0).load(
    "dane_delta/zamowienia_delta"
).filter("klient_id = 1")
po = tabela_zamowien.toDF().filter("klient_id = 1")

print("Statusy przed MERGE:")
przed.groupBy("status").count().show()
print("Statusy po MERGE:")
po.groupBy("status").count().show()
```

To dokladnie wzorzec uzywany w prawdziwych potokach danych: 'oznacz anulowane zamowienia klienta' bez przepisywania calej tabeli od zera, z pelna historia zmian dostepna do audytu.

</details>

## Co dalej?

W **Module 11** zbudujemy pełny obraz **architektury Lakehouse**: warstwy
bronze/silver/gold (Medallion Architecture), różnice między data lake, data warehouse i
lakehouse, oraz podstawy Unity Catalog.